In [52]:
import pandas as pd
import sklearn
from sklearn.preprocessing import OneHotEncoder

In [85]:
df_jan = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet')
df_feb = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet')

In [19]:
df_full = pd.concat([df_jan, df_feb])

## Question 1: Number of Columns

Based on the command below, the answer is 19 columns

In [42]:
df_jan.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'airport_fee', 'duration'],
      dtype='object')

## Question 2: Duration Standard Deviation

In [29]:
df_jan['duration'] =  pd.to_datetime(df_jan['tpep_dropoff_datetime']) - pd.to_datetime(df_jan['tpep_pickup_datetime'])
df_jan['duration'] = df_jan['duration'].dt.total_seconds() / 60

In [30]:
df_jan['duration'].std()

42.594351241920904

The answer is 42.59

## Question 3: Removing Outliers

In [31]:
Q1 = df_jan['duration'].quantile(0.25)
Q3 = df_jan['duration'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

In [33]:
upper_bound

35.075

In [38]:
df_jan_filter = df_jan[(df_jan['duration'] <= 60) & (df_jan['duration'] >= 1)]

In [39]:
len(df_jan_filter) / len(df_jan)

0.9812202822125979

98% of the data is left after we filter based on the question suggested

## Question 4: Creating Feature Matrix

In [89]:
encoder = OneHotEncoder(handle_unknown='ignore')
X = encoder.fit_transform(df_jan_filter[['PULocationID', 'DOLocationID']])
y = df_jan_filter['duration']

In [90]:
X

<3009173x515 sparse matrix of type '<class 'numpy.float64'>'
	with 6018346 stored elements in Compressed Sparse Row format>

As the previous cell demonstrates, there are 515 columns

## Question 5: RMSE Training Loss

In [79]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

model = LinearRegression(fit_intercept=True)
model.fit(X, y)

train_pred = model.predict(X)
train_rmse = mean_squared_error(y, train_pred, squared=False)

In [80]:
print(train_rmse)

7.649261027909014


The training loss is 7.65.

## Question 6: RMSE Validation Loss

In [91]:
df_feb['duration'] =  pd.to_datetime(df_feb['tpep_dropoff_datetime']) - pd.to_datetime(df_feb['tpep_pickup_datetime'])
df_feb['duration'] = df_feb['duration'].dt.total_seconds() / 60


df_feb_filter = df_feb[(df_feb['duration'] <= 60) & (df_feb['duration'] >= 1)]

In [92]:
X_val = encoder.transform(df_feb_filter[['PULocationID', 'DOLocationID']])
y_val = df_feb_filter['duration']

In [93]:
val_pred = model.predict(X_val)
val_rmse = mean_squared_error(y_val, val_pred, squared=False)

In [94]:
val_rmse

7.811832536183877

RMSE Validation loss is 7.81